# 05 · Multi-agent
**The limitation this fixes:** one agent with ten tools gets confused, and every call pays for every tool's description.

| Feature | Framework piece |
|---|---|
| Specialists | each one is its own `create_agent` |
| Supervisor calls specialists | specialist wrapped in a `@tool` |
| Doctor's pre-visit brief | `response_format=PreVisitBrief` (structured output) |
| Safety check that can end the turn | a node in a `StateGraph` + conditional edge |
| Nested traces | automatic in LangSmith |

In [ ]:
# Setup: the same cell at the top of every notebook
from dotenv import load_dotenv
load_dotenv("../.env", override=True)   # OPENAI_API_KEY + LANGSMITH_* (set LANGSMITH_TRACING=true to see traces)

MODEL = "openai:gpt-4o-mini"           # swap the provider here, e.g. "anthropic:claude-sonnet-4-6"

## What we built so far

In [ ]:
from common import (CareContext, RAG_PROMPT, get_care_plan, log_reading, log_symptom,
                    get_recent_readings, build_leaflet_search, list_free_slots, book_appointment,
                    patients_needing_attention, dose_guard, ask, resume, show_state)
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langgraph.checkpoint.memory import InMemorySaver

search_documents = build_leaflet_search()

## 1. Specialists: small agents with few tools each

In [ ]:
records_agent = create_agent(
    MODEL, tools=[get_care_plan, log_reading, log_symptom, get_recent_readings, search_documents,
                  patients_needing_attention],
    system_prompt=RAG_PROMPT + "\nYou handle care plans, readings, symptoms and documents.",
    context_schema=CareContext)

booking_agent = create_agent(
    MODEL, tools=[list_free_slots, book_appointment],
    system_prompt="You book appointments. If no exact slot is given, list the free slots.",
    context_schema=CareContext)

Wrap each one as a tool. The docstring tells the supervisor **when** to use it. We pass the same context down.

In [ ]:
def run(agent, request, runtime):
    result = agent.invoke({"messages": [{"role": "user", "content": request}]}, context=runtime.context)
    return result["messages"][-1].content

@tool
def ask_records_agent(request: str, runtime: ToolRuntime[CareContext]) -> str:
    """Delegate anything about care plans, readings, symptoms, leaflets or check-in notes."""
    return run(records_agent, request, runtime)

@tool
def ask_booking_agent(request: str, runtime: ToolRuntime[CareContext]) -> str:
    """Delegate anything about appointment slots and booking."""
    return run(booking_agent, request, runtime)

## 2. Structured output: the doctor's pre-visit brief
Instead of free text, the agent must fill in a Pydantic object.

In [ ]:
from pydantic import BaseModel, Field

class PreVisitBrief(BaseModel):
    condition: str
    latest_readings: list[str]
    trend: str = Field(description="improving, stable or worsening")
    concerns: list[str]
    questions_for_visit: list[str]

brief_agent = create_agent(MODEL, tools=[get_care_plan, get_recent_readings, search_documents],
                           system_prompt="Prepare a factual pre-visit brief for a doctor. No diagnosis.",
                           response_format=PreVisitBrief, context_schema=CareContext)

@tool
def write_previsit_brief(runtime: ToolRuntime[CareContext]) -> str:
    """For doctors: a structured brief of this patient before their visit."""
    result = brief_agent.invoke({"messages": [{"role": "user", "content": "Brief me on this patient."}]},
                                context=runtime.context)
    return result["structured_response"].model_dump_json(indent=2)

## 3. The supervisor: an agent whose tools are agents
Its prompt is short and strict: it knows nothing itself, so it must **always delegate**. Without that, it happily answers from general knowledge and skips the specialists (and their safety checks).

In [ ]:
SUPERVISOR_PROMPT = """You are the front desk of a care team. You know NOTHING about the patient
or our documents yourself, so never answer from your own knowledge.
For every request, call a specialist and pass on the patient's words exactly, including any numbers:
- ask_records_agent: care plans, targets, medicines, readings (including new ones to log), symptoms,
  how-to questions and check-in notes
- ask_booking_agent: appointment slots and booking
- write_previsit_brief: a doctor's brief before a visit
Then relay the specialist's answer. Never give or change medicine doses."""

supervisor = create_agent(
    MODEL, tools=[ask_records_agent, ask_booking_agent, write_previsit_brief],
    system_prompt=SUPERVISOR_PROMPT,
    middleware=[dose_guard],   # same code-level guard as notebook 02
    context_schema=CareContext)

## 4. The top-level graph: a safety node with a veto
`create_agent` builds a graph for us. Here we build one by hand with `StateGraph`: **safety → supervisor**,
and the safety node can send the conversation straight to `END`.

In [ ]:
import re
from langchain.messages import AIMessage
from langgraph.graph import StateGraph, MessagesState, START, END

EMERGENCY = re.compile(r"chest pain|can'?t breathe|unconscious|stroke|suicid", re.I)   # CUSTOM: triage policy
EMERGENCY_REPLY = "This could be an emergency. Call emergency services now. I have alerted your care team."

def safety(state: MessagesState):
    if EMERGENCY.search(state["messages"][-1].content):
        return {"messages": [AIMessage(EMERGENCY_REPLY)]}
    return {}

def after_safety(state: MessagesState):
    return END if isinstance(state["messages"][-1], AIMessage) else "supervisor"

graph = StateGraph(MessagesState, context_schema=CareContext)
graph.add_node("safety", safety)
graph.add_node("supervisor", supervisor)
graph.add_edge(START, "safety")
graph.add_conditional_edges("safety", after_safety, ["supervisor", END])
graph.add_edge("supervisor", END)
care_team = graph.compile(checkpointer=InMemorySaver())

In [ ]:
from IPython.display import Image
Image(care_team.get_graph(xray=True).draw_mermaid_png())

## Demo

In [ ]:
ask(care_team, "What's my target blood pressure, and how should I measure it at home?", thread="mixed")
ask(care_team, "Can I book the first free slot?", thread="mixed")

## ❌ Failure case: an emergency never reaches the supervisor
🧾 **Receipt:** only 2 messages in state, no model was called.

In [ ]:
ask(care_team, "I have crushing chest pain", thread="emergency")
show_state(care_team, thread="emergency")

## The doctor's brief

In [ ]:
dr_lee = CareContext(user_id="dr_lee", role="doctor")
ask(care_team, "Prepare my pre-visit brief for this patient", thread="brief", context=dr_lee)

👉 In LangSmith, open the trace: supervisor → `ask_records_agent` → records agent → `search_documents`, all nested.

## Where this lives in the backend
- `app/multi_agent.py: specialists, supervisor and the `StateGraph``
- `langgraph.json: served as the `care_team` graph, and Studio draws it`

Same primitives, just split into files. The notebook never imports `app/`.